In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_heads=4, dim_feedforward=None):
        super().__init__()
        if dim_feedforward is None:
            dim_feedforward = d_model * 4
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens, key_padding_mask=None):
        return self.final_norm(self.encoder(tokens, src_key_padding_mask=key_padding_mask))

In [3]:
class WholeBrainPatchEmbed3D(nn.Module):
    def __init__(self, brain_size=112, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = brain_size // patch_size
        self.n_tokens = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, :, :]
        occupancy = F.max_pool3d((volume.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool()
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class WholeBrainTransformerBranch(nn.Module):
    def __init__(self, brain_size=112, patch_size=8, d_model=32, n_layers=2, n_heads=4):
        super().__init__()
        self.patch_embed = WholeBrainPatchEmbed3D(brain_size, patch_size, d_model)
        self.encoder = TransformerEncoder(d_model, n_layers, n_heads)

    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        key_padding_mask = ~valid
        tokens = self.encoder(tokens, key_padding_mask=key_padding_mask)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class WholeBrainTransformerModel(nn.Module):
    def __init__(self, brain_size=112, patch_size=8, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4):
        super().__init__()
        self.branch = WholeBrainTransformerBranch(brain_size, patch_size, d_model, n_layers, n_heads)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, volume):
        pooled = self.branch(volume)
        return self.classifier(self.dropout(pooled))


class MultimodalWholeBrainTransformerModel(nn.Module):
    def __init__(self, brain_size=112, patch_size=8, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4):
        super().__init__()
        self.mri_branch = WholeBrainTransformerBranch(brain_size, patch_size, d_model, n_layers, n_heads)
        self.pet_branch = WholeBrainTransformerBranch(brain_size, patch_size, d_model, n_layers, n_heads)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_vol, pet_vol):
        mri_pooled = self.mri_branch(mri_vol)
        pet_pooled = self.pet_branch(pet_vol)
        return self.classifier(self.dropout(torch.cat([mri_pooled, pet_pooled], dim=1)))

In [4]:
COHORT_CSV       = "D:/mamba_model/thesis_cohort_final.csv"
WB_MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_mri_aug"
WB_PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_pet_aug"
CKPT_DIR         = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

BRAIN_SIZE = 120

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class WholeBrainDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        vol = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), key


class MultimodalWholeBrainDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_vol = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_vol = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(mri_vol).unsqueeze(0), torch.from_numpy(pet_vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), mri_key

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for vol, labels, _ in loader:
        vol, labels = vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for vol, labels, _ in loader:
            vol, labels = vol.to(device), labels.to(device)
            out = model(vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_vol, pet_vol, labels, _ in loader:
        mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_vol, pet_vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_vol, pet_vol, labels, _ in loader:
            mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
            out = model(mri_vol, pet_vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                mri_vol, pet_vol = mri_vol.to(device), pet_vol.to(device)
                bs = mri_vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_vol, pet_vol)
            else:
                vol, labels, _ = batch
                vol = vol.to(device)
                bs = vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(vol)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                inputs = (mri_vol[:1].to(device), pet_vol[:1].to(device))
            else:
                vol, labels, _ = batch
                inputs = (vol[:1].to(device),)
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(brain_size=BRAIN_SIZE, d_model=32, n_layers=2, n_heads=4, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({epoch_time:.1f}s)")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
BATCH_SIZE = 4
wb_mri_train = DataLoader(WholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mri_val   = DataLoader(WholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mri_test  = DataLoader(WholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== TRANSFORMER, WHOLE-BRAIN MRI-ONLY (token-matched): 3-seed run ===")
transformer_wb_mri_results = [run_one_seed(s, WholeBrainTransformerModel, wb_mri_train, wb_mri_val, wb_mri_test,
                                             False, "transformer_wholebrain_mri") for s in [1, 7, 123]]

=== TRANSFORMER, WHOLE-BRAIN MRI-ONLY (token-matched): 3-seed run ===

--- Seed 1 ---
Epoch 1: train_loss=0.7028 val_loss=0.6923 val_acc=0.5000 (261.0s)
Epoch 2: train_loss=0.7028 val_loss=0.6959 val_acc=0.5000 (157.6s)
Epoch 3: train_loss=0.6921 val_loss=0.6909 val_acc=0.5000 (146.5s)
Epoch 4: train_loss=0.6971 val_loss=0.6898 val_acc=0.5000 (151.3s)
Epoch 5: train_loss=0.6902 val_loss=0.6892 val_acc=0.5238 (155.4s)
Epoch 6: train_loss=0.6849 val_loss=0.6888 val_acc=0.5000 (134.3s)
Epoch 7: train_loss=0.6853 val_loss=0.6890 val_acc=0.5000 (128.6s)
Epoch 8: train_loss=0.6822 val_loss=0.6851 val_acc=0.5238 (136.5s)
Epoch 9: train_loss=0.6760 val_loss=0.6842 val_acc=0.5714 (144.0s)
Epoch 10: train_loss=0.6794 val_loss=0.6856 val_acc=0.5238 (131.5s)
Epoch 11: train_loss=0.6751 val_loss=0.6855 val_acc=0.5000 (56.6s)
Epoch 12: train_loss=0.6646 val_loss=0.6860 val_acc=0.5238 (14.6s)
Epoch 13: train_loss=0.6598 val_loss=0.6939 val_acc=0.5000 (14.8s)
Epoch 14: train_loss=0.6525 val_loss=0.694

In [9]:
wb_pet_train = DataLoader(WholeBrainDataset(X_train, y_train, WB_PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
wb_pet_val   = DataLoader(WholeBrainDataset(X_val, y_val, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
wb_pet_test  = DataLoader(WholeBrainDataset(X_test, y_test, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== TRANSFORMER, WHOLE-BRAIN PET-ONLY (token-matched): 3-seed run ===")
transformer_wb_pet_results = [run_one_seed(s, WholeBrainTransformerModel, wb_pet_train, wb_pet_val, wb_pet_test,
                                             False, "transformer_wholebrain_pet") for s in [1, 7, 123]]

=== TRANSFORMER, WHOLE-BRAIN PET-ONLY (token-matched): 3-seed run ===

--- Seed 1 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


Epoch 1: train_loss=0.7108 val_loss=0.6933 val_acc=0.5000 (259.6s)
Epoch 2: train_loss=0.7048 val_loss=0.6967 val_acc=0.4762 (80.1s)
Epoch 3: train_loss=0.6921 val_loss=0.6885 val_acc=0.5238 (16.0s)
Epoch 4: train_loss=0.6963 val_loss=0.6839 val_acc=0.5952 (14.9s)
Epoch 5: train_loss=0.6899 val_loss=0.6790 val_acc=0.6429 (14.5s)
Epoch 6: train_loss=0.6782 val_loss=0.6730 val_acc=0.6190 (14.5s)
Epoch 7: train_loss=0.6809 val_loss=0.6741 val_acc=0.5476 (14.6s)
Epoch 8: train_loss=0.6746 val_loss=0.6568 val_acc=0.6667 (15.0s)
Epoch 9: train_loss=0.6664 val_loss=0.6464 val_acc=0.6667 (16.9s)
Epoch 10: train_loss=0.6642 val_loss=0.6450 val_acc=0.6905 (15.3s)
Epoch 11: train_loss=0.6644 val_loss=0.6350 val_acc=0.6905 (14.6s)
Epoch 12: train_loss=0.6447 val_loss=0.6418 val_acc=0.6905 (14.5s)
Epoch 13: train_loss=0.6517 val_loss=0.6587 val_acc=0.6190 (14.6s)
Epoch 14: train_loss=0.6482 val_loss=0.6339 val_acc=0.6667 (14.8s)
Epoch 15: train_loss=0.6466 val_loss=0.6348 val_acc=0.6429 (16.3s)
Epo

In [10]:
wb_mm_train = DataLoader(MultimodalWholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mm_val   = DataLoader(MultimodalWholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mm_test  = DataLoader(MultimodalWholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== TRANSFORMER, WHOLE-BRAIN MULTIMODAL (token-matched): 3-seed run ===")
transformer_wb_mm_results = [run_one_seed(s, MultimodalWholeBrainTransformerModel, wb_mm_train, wb_mm_val, wb_mm_test,
                                            True, "transformer_wholebrain_mm") for s in [1, 7, 123]]

=== TRANSFORMER, WHOLE-BRAIN MULTIMODAL (token-matched): 3-seed run ===

--- Seed 1 ---
Epoch 1: train_loss=0.7136 val_loss=0.6879 val_acc=0.5238 (310.8s)
Epoch 2: train_loss=0.6969 val_loss=0.6914 val_acc=0.5000 (316.7s)
Epoch 3: train_loss=0.6925 val_loss=0.6867 val_acc=0.4762 (307.1s)
Epoch 4: train_loss=0.6962 val_loss=0.6918 val_acc=0.5000 (303.8s)
Epoch 5: train_loss=0.6869 val_loss=0.6814 val_acc=0.6429 (308.8s)
Epoch 6: train_loss=0.6835 val_loss=0.6783 val_acc=0.6429 (290.5s)
Epoch 7: train_loss=0.6811 val_loss=0.6894 val_acc=0.5238 (323.6s)
Epoch 8: train_loss=0.6740 val_loss=0.6724 val_acc=0.6667 (314.7s)
Epoch 9: train_loss=0.6781 val_loss=0.6697 val_acc=0.6429 (314.9s)
Epoch 10: train_loss=0.6773 val_loss=0.6702 val_acc=0.6190 (308.7s)
Epoch 11: train_loss=0.6767 val_loss=0.6657 val_acc=0.6429 (292.3s)
Epoch 12: train_loss=0.6542 val_loss=0.6608 val_acc=0.6429 (291.3s)
Epoch 13: train_loss=0.6495 val_loss=0.6499 val_acc=0.6667 (301.2s)
Epoch 14: train_loss=0.6413 val_loss=

In [11]:
def summarize(results, name):
    accs, tprs, tnrs = [r["acc"] for r in results], [r["tpr"] for r in results], [r["tnr"] for r in results]
    print(f"{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")

print("=== Transformer, whole-brain (token-matched) ===")
summarize(transformer_wb_mri_results, "MRI-only")
summarize(transformer_wb_pet_results, "PET-only")
summarize(transformer_wb_mm_results, "Multimodal")

print("\n=== Compare to v4_wholebrain (Mamba) ===")
print("MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%")
print("PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%")
print("Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%")

=== Transformer, whole-brain (token-matched) ===
MRI-only: Acc=48.4±1.4% | TPR=52.4±8.2% | TNR=44.4±7.3%
PET-only: Acc=63.5±1.4% | TPR=58.7±2.7% | TNR=68.3±2.7%
Multimodal: Acc=60.3±6.9% | TPR=61.9±8.2% | TNR=58.7±22.0%

=== Compare to v4_wholebrain (Mamba) ===
MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%
PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%
Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%
